# 2 · A first learned upscaler

The smallest neural network that counts as a learned upscaler, trained on the patches
from `0_data_preprocess.ipynb`. Steps:

1. **Data** - load the patch pairs.
2. **Model** - a few convolutions that add detail on top of bicubic.
3. **Train** - and compare with bicubic after every epoch.
4. **Speed** - is it fast enough for a live stream?
5. **Save** - write a checkpoint and check it loads back the same.
6. **Export** - write it as ONNX for the browser benchmark.

Needs: `0_data_preprocess.ipynb`. The bar to beat comes from `1_upscale_dummy.ipynb`.
Results: `summary.md`, "Learned model".

In [ ]:
"""Settings."""

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

from helpers import metrics, models, speed, training, webexport
from helpers.training import DATA_DIR, DEVICE

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_nn.pt"
FRAME_DIR = DATA_DIR / "synthetic"

CHANNELS = 64          # width of the first convolution
EPOCHS = 12
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0

BENCH_RESOLUTIONS = [(480, 270), (640, 360), (960, 540), (1280, 720)]   # LR input sizes
BENCH_LR_SIZE = (320, 180)
BENCH_BATCHES = [1, 2, 4, 8, 16]

training.seed_everything(SEED)

## 1. Data

`train` is what the model learns from (with random flips and rotations, which are free
extra examples). `val` is held out: every score below is measured on it.

The LR patch has a halo of extra pixels around it, so the model's output is a bit too
large. `trim` cuts that border off before the loss and the scores.

In [ ]:
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE)
trim = training.trimmer(manifest)

lr_batch, hr_batch = next(iter(train_loader))
print("lr batch", tuple(lr_batch.shape), " hr batch", tuple(hr_batch.shape))
training.show_patches({"LR (with halo)": lr_batch[:4], "HR": hr_batch[:4]})

## 2. Model

Shaped like ESPCN: convolutions at **low** resolution, then one `PixelShuffle` that
rearranges channels into a larger image. Working at low resolution is 4x cheaper than
upscaling first.

It predicts a **correction on top of bicubic**. The last layer starts at zero, so the
untrained model *is* bicubic, and every dB it gains in training is real improvement.

Its **halo** is 4: the 5x5 reaches 2 pixels, then two 3x3s reach 1 each.

In [ ]:
class ResidualUpscaler(nn.Module):
    def __init__(self, scale, channels=64):
        super().__init__()
        self.scale = scale
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels // 2, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.residual = models.zero_start(nn.Conv2d(channels // 2, 3 * scale * scale, 3, padding=1))
        self.shuffle = nn.PixelShuffle(scale)
        self.halo = 4

    def forward(self, x):
        return models.bicubic(x, self.scale) + self.shuffle(self.residual(self.features(x)))


model = ResidualUpscaler(SCALE, CHANNELS).to(DEVICE)
print(model)
print(f"{models.count_parameters(model):,} parameters")

In [ ]:
"""Before training, the model must score exactly like bicubic."""

bicubic = models.Resize(SCALE).to(DEVICE)
baseline = training.evaluate(bicubic, val_loader, trim)
start = training.evaluate(model, val_loader, trim)
metrics.print_table({"bicubic": baseline, "untrained model": start})
assert abs(start["rgb"] - baseline["rgb"]) < 1e-6, "the model should start as bicubic"

## 3. Train

L1 loss (mean absolute error): it gives sharper results than squared error, which
rewards blurry averages. After every epoch we score the validation split and keep the
best epoch by RGB PSNR.

In [ ]:
history, best = training.train(model, train_loader, val_loader, trim, EPOCHS,
                               learning_rate=LEARNING_RATE, select="rgb", print_every=1)
training.plot_history(history, baseline, key="rgb")

final = training.evaluate(model, val_loader, trim)
metrics.print_table({"bicubic": baseline, "model": final},
                    extra=lambda name, s: f"   {s['rgb'] - baseline['rgb']:+.2f} dB rgb vs bicubic")

### What it looks like

Check text lines and window edges - that is what a desktop stream is made of.

In [ ]:
lr, hr = next(iter(val_loader))
lr, hr = lr[:4].to(DEVICE), hr[:4].to(DEVICE)
with torch.no_grad():
    training.show_patches({
        "nearest": trim(F.interpolate(lr, scale_factor=SCALE, mode="nearest")),
        "bicubic": trim(models.bicubic(lr, SCALE)),
        "model": trim(model.eval()(lr)),
        "target": hr,
    })

## 4. Speed

For a live stream, speed is half the result: a model that misses the frame time is not
usable, however good it looks. A batch of one is the honest number - frames arrive one
at a time. A larger batch can lower the cost per frame, but each frame then waits for
the others.

In [ ]:
"""One frame at a time, at the sizes a client asks for."""

model.eval()
print(f"{'output':>12}{'ms/frame':>10}   fits")
with torch.no_grad():
    for width, height in BENCH_RESOLUTIONS:
        x = torch.rand(1, 3, height, width, device=DEVICE)
        ms = speed.time_ms(lambda: model(x), seconds=1.0)
        print(f"{f'{width * SCALE}x{height * SCALE}':>12}{ms:10.1f}   {speed.frame_budget(ms)}")

In [ ]:
"""What a batch buys at 320x180 -> 640x360."""

per_frame = {}
with torch.no_grad():
    for batch in BENCH_BATCHES:
        x = torch.rand(batch, 3, BENCH_LR_SIZE[1], BENCH_LR_SIZE[0], device=DEVICE)
        per_frame[batch] = speed.time_ms(lambda: model(x), seconds=1.0) / batch

print(f"{'batch':>5}{'ms/frame':>10}{'vs batch 1':>12}")
for batch, ms in per_frame.items():
    print(f"{batch:5}{ms:10.2f}{per_frame[1] / ms:11.2f}x")

## 5. Save and reload

The checkpoint holds the weights *and* everything needed to rebuild the model. We load
it into a fresh model and check it computes the same thing.

In [ ]:
training.save_checkpoint(CHECKPOINT, model, architecture="ResidualUpscaler", scale=SCALE,
                         channels=CHANNELS, halo=model.halo, epochs=EPOCHS,
                         best_epoch=best["epoch"], val=final, val_bicubic=baseline,
                         degradation=manifest["degradation"])

checkpoint = training.load_checkpoint(CHECKPOINT)
loaded = ResidualUpscaler(checkpoint["scale"], checkpoint["channels"]).to(DEVICE)
loaded.load_state_dict(checkpoint["state_dict"])
training.assert_same_model(model, loaded)
print("reloaded model is identical")

### A whole frame

The model is fully convolutional, so it runs on a picture of any size, not only patches.

In [ ]:
truth = Image.open(sorted(FRAME_DIR.glob("*.png"))[0]).convert("RGB")
truth = truth.crop((0, 0, truth.width - truth.width % SCALE, truth.height - truth.height % SCALE))
small = truth.resize((truth.width // SCALE, truth.height // SCALE), Image.BICUBIC)

with torch.no_grad():
    upscaled = loaded.eval()(training.to_tensor(small))
target = training.to_tensor(truth)
metrics.print_table({"bicubic": metrics.report(models.bicubic(training.to_tensor(small), SCALE), target),
                     "model": metrics.report(upscaled, target)})
Image.fromarray(training.to_image(upscaled)).save(training.SAMPLE_DIR / "upscaled_nn.png")

box = (slice(0, 200), slice(0, 320))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
panels = {"bicubic": np.asarray(small.resize(truth.size, Image.BICUBIC)),
          "model": training.to_image(upscaled), "target": np.asarray(truth)}
for ax, (name, image) in zip(axes, panels.items()):
    ax.imshow(image[box], interpolation="nearest")
    ax.set_title(name)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 6. Export for the browser

The bicubic part of this model is `F.interpolate`, which becomes a `Resize` node in
ONNX. `Resize` is outside the operators the browser runs fast (the summary line below
flags it), so this model is slow in the browser. `3_upscale_web.ipynb` fixes that.

In [ ]:
size = webexport.TILE_STEP + 2 * checkpoint["halo"]
info = webexport.export(loaded, f"upscale_nn_tile{webexport.TILE_STEP}.onnx", size,
                        label="nn - bicubic residual (has Resize)", halo=checkpoint["halo"])
webexport.check_graph(info["path"], loaded, np.random.rand(1, 3, size, size).astype(np.float32))
ms = webexport.time_graph(info["path"], size)
webexport.annotate(info, cpu_ms=round(ms, 2))
print(webexport.summary_line(info, ms))
print("compare in a browser: uv run src/upscale/benchmark/main.py")